# Churn Prediction — EDA & Feature Development

Goal: understand which customer attributes drive churn and validate the engineered features before modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

df = pd.read_csv('../data/churn_train.csv', parse_dates=['signup_date'])
print(df.shape)
df.head()

In [ ]:
# target distribution
churn_rate = (df['Churn'] == 'Yes').mean()
print(f'Churn rate: {churn_rate:.1%}')
print(f'Retained:   {1-churn_rate:.1%}')

# class imbalance — noting this for model config
df['Churn'].value_counts().plot(kind='bar', color=['#2c3e50','#e74c3c'])
plt.title('Churn Distribution')
plt.xticks(rotation=0)
plt.tight_layout()

## Churn by Contract Type

Hypothesis: month-to-month customers churn significantly more — lower switching costs.

In [ ]:
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean())
churn_by_contract.sort_values(ascending=False).plot(kind='bar', color='#c0392b')
plt.title('Churn Rate by Contract Type')
plt.ylabel('Churn Rate')
plt.xticks(rotation=15)
plt.tight_layout()
print(churn_by_contract.round(3))

## Monthly Charges vs Churn

Higher charges = higher churn risk, but the relationship isn't linear.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

df.boxplot('MonthlyCharges', by='Churn', ax=ax1)
ax1.set_title('Monthly Charges by Churn')
ax1.set_xlabel('Churn')

df.hist('tenure', by=df['Churn'], ax=ax2, bins=20, color=['#2c3e50','#c0392b'], alpha=0.7)
plt.suptitle('')
plt.tight_layout()

## Feature Correlations with Churn

Checking raw numeric features before engineering.

In [ ]:
df['churn_binary'] = (df['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure','MonthlyCharges','TotalCharges','NumSupportCalls','NumProductChanges','DaysSinceLastLogin']
correlations = df[corr_cols + ['churn_binary']].corr()['churn_binary'].drop('churn_binary').sort_values()

correlations.plot(kind='barh', color=correlations.map(lambda x: '#c0392b' if x > 0 else '#2c3e50'))
plt.title('Correlation with Churn (raw features)')
plt.axvline(0, color='gray', linestyle='--')
plt.tight_layout()
print(correlations.round(3))

## Engineered Feature Validation

Testing that engineered features improve signal over raw features.

In [ ]:
# avg monthly charge = stronger signal than raw MonthlyCharges or tenure alone
df['avg_monthly_charge'] = df['TotalCharges'] / (df['tenure'] + 1)
df['support_call_rate'] = df['NumSupportCalls'] / (df['tenure'] + 1)
df['near_renewal'] = ((df['tenure'] % 12) >= 10).astype(int)

eng_cols = ['avg_monthly_charge','support_call_rate','near_renewal']
eng_corr = df[eng_cols + ['churn_binary']].corr()['churn_binary'].drop('churn_binary').sort_values()

print('Engineered feature correlations:')
print(eng_corr.round(3))

# Compare: avg_monthly_charge vs raw MonthlyCharges correlation
print(f"\nRaw MonthlyCharges corr: {df['MonthlyCharges'].corr(df['churn_binary']):.3f}")
print(f"Avg monthly charge corr: {df['avg_monthly_charge'].corr(df['churn_binary']):.3f}")

## Support Call Analysis

Support calls are a strong churn signal — customers who call repeatedly are at high risk.

In [ ]:
churn_by_calls = df.groupby('NumSupportCalls')['churn_binary'].mean()
churn_by_calls[:8].plot(marker='o', color='#c0392b')
plt.title('Churn Rate by Number of Support Calls')
plt.xlabel('Support Calls')
plt.ylabel('Churn Rate')
plt.tight_layout()

print('Churn rate by support call count:')
print(churn_by_calls[:8].round(3))

## Observations

- Month-to-month contracts: ~3x higher churn rate than two-year contracts
- Avg monthly charge is a stronger predictor than raw MonthlyCharges (normalizes for tenure)
- Support call rate (calls/month) is the highest-correlation engineered feature
- Customers near contract renewal (months_to_renewal ≤ 2) show elevated churn — intervention window
- Tenure < 12 months = highest-risk segment regardless of contract type

These observations directly inform the feature engineering in `src/features.py`.